# Session 7 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold

# Short-stay listings with a price, as in the lecture notebooks of this session
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
cols = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "room_type", "district",
        "number_of_reviews", "review_scores_rating", "availability_365"]
X = pd.get_dummies(bnb[cols], columns=["room_type", "district"], dtype=int)
y, price, hosts = np.log(bnb["price"]), bnb["price"], bnb["host_id"]
print(X.shape, "listings x columns,", hosts.nunique(), "hosts")

## Part 1 · An honest estimate: data splits, cross-validation and leakage

### 1.1 · Who is left out of a bootstrap sample?

*Paper, 10 min*

> The probability that a given row is never drawn in n draws with replacement. What does it approach for large n?

*Source:* [James et al., *An Introduction to Statistical Learning*, Chapter 5, Exercise 2](https://www.statlearning.com/)

Your answer:

### 1.2 · The same host on both sides

*Paper, 5 min*

> One host owns 40 near-identical flats. What happens to the score with random folds?

*Source:* original exercise for this course; background: [scikit-learn: Cross-validation, group k-fold](https://scikit-learn.org/stable/modules/cross_validation.html)

Your answer:

### 1.3 · Which of these workflows leak?

*Predict, 10 min*

> Four workflows for the Berlin price model, each evaluated with 5-fold cross-validation afterwards. For each, decide whether it leaks, and rank them by how much they would inflate the score.
>
> (a) `StandardScaler().fit_transform(X)` on all rows, then `cross_val_score(Ridge(), ...)`.
> (b) The mean log price of each neighbourhood, computed with `groupby` on all rows and added as a column.
> (c) `np.log` of the price as the target, computed before the split.
> (d) The 20 amenities most correlated with the price, chosen on all rows, then cross-validated.

*Source:* original exercise for this course; background: [scikit-learn: Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html)

Your answer:

### 1.4 · Perfect accuracy on pure noise

*Code, 20 min*

> 50 rows, 1,000 random features, a random binary target. Select the 10 features most correlated with the target on all the data, then cross-validate a classifier.

*Source:* [Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, §7.10.2 "The wrong and right way to do cross-validation"](https://hastie.su.domains/ElemStatLearn/)

### 1.5 · Is gradient boosting really better for new hosts?

*Code, 25 min, *Stretch**

> Use the setup data (`X`, `y`, `price`, `hosts`). Lock away 20 % of the hosts with `GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)`. Fit a `Ridge(alpha=10)` (in a pipeline after a `SimpleImputer(strategy="median")` and a `StandardScaler`; some columns have missing values) and a `HistGradientBoostingRegressor(random_state=0)` on the other hosts and compute the absolute error in euros of both models on the locked hosts.
>
> Is the difference in MAE between the two models larger than chance? Compute a 95 % cluster-bootstrap interval for the *difference* (resample hosts, and use the same resampled hosts for both models). What do you conclude?

*Source:* original exercise for this course; background: Koehn (2004), [Statistical significance tests for machine translation evaluation](https://aclanthology.org/W04-3250/) (paired bootstrap)

## Part 2 · Choosing settings: regularisation and tuning

### 2.1 · Ridge as λ → ∞

*Predict, 5 min*

> What happens to the coefficients of a ridge regression when the penalty λ (scikit-learn: `alpha`) grows without limit? Sketch why the lasso sets coefficients exactly to zero and ridge does not.

*Source:* [*The Elements of Statistical Learning*, §3.4, Figure 3.11](https://hastie.su.domains/ElemStatLearn/)

Your answer:

### 2.2 · Sketch the learning curves

*Paper, 10 min*

> Sketch training and validation error against training size for an underfitting and an overfitting model. Then compare with the real curves.

*Source:* original exercise for this course; background: [scikit-learn: Validation and learning curves](https://scikit-learn.org/stable/modules/learning_curve.html)

Your answer:

### 2.3 · The public leaderboard trap

*Code, 15 min*

> A test set of 200 random labels, split into a public and a private half. Generate 100 random submissions and choose the best on the public half. How good is it on the private half?

*Source:* [Blum & Hardt (2015), "The Ladder: a reliable leaderboard for machine learning competitions"](https://arxiv.org/abs/1502.04585)

### 2.4 · The test set used fifty times

*Class, 5 min*

> A team evaluates on its test set after every change, fifty times in total. Is it still a test set?

*Source:* [Dwork et al. (2015), "The reusable holdout", *Science*](https://doi.org/10.1126/science.aaa9375) ([arXiv](https://arxiv.org/abs/1506.02629))

Your answer:

### 2.5 · How many neighbours for a price?

*Code, 25 min, *Stretch**

> Use the setup data. Put a `SimpleImputer(strategy="median")`, a `StandardScaler` and a `KNeighborsRegressor` in a pipeline and tune `n_neighbors` (for example 5, 10, 20, 50, 100) with `GridSearchCV` and host-grouped folds (`GroupKFold(5)`, pass `groups=hosts` to `fit`). Report `best_params_` and `best_score_` (R² on the log price).
>
> Then estimate the tuned procedure without optimism with nested cross-validation (inner `GroupKFold(3)`, outer `GroupKFold(5)`). Is the nested estimate lower than `best_score_`? Is k-NN better or worse than the ridge model of the lecture (host-grouped R² about 0.6 on more columns)?

*Source:* original exercise for this course; background: [scikit-learn: Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)